In [ ]:
import os
import pandas as pd
import requests
import time
import random
from tqdm import tqdm

In [6]:
df = pd.read_csv("ISGD/Attributes.csv")

# Normalize column names
df.columns = df.columns.str.strip().str.lower()

# Process only a small subset first to keep prompt generation fast
df = df.head(100).copy()

df.head()

,image_id,attractive,blurry_image,sharp_jawline,high_cheekbones,smiling,bald,receeding_hairline,long_hair,curly_hair,...,big_lips,sharp_nose,adult,old,mouth_open,male,double_chin,veil,wrinkle,chubby
0,00001.jpg,0,0,0,1,1,0,0,0,0,...,0,0,1,0,0,1,0,0,1,1
1,00002.jpg,1,0,0,1,1,0,0,1,0,...,0,0,1,0,1,0,0,0,0,0
2,00003.jpg,0,0,0,0,0,0,0,0,0,...,0,0,1,0,1,1,0,0,0,0
3,00004.jpg,1,0,1,0,0,0,0,1,0,...,1,0,0,0,0,0,0,0,0,0
4,00005.jpg,0,0,0,0,0,0,0,0,0,...,0,0,1,0,1,1,0,0,1,1


In [7]:
ATTRIBUTE_MAP = {
    "attractive": "an attractive face",
    "blurry_image": "a slightly blurry image",
    "sharp_jawline": "a sharp jawline",
    "high_cheekbones": "high cheekbones",
    "smiling": "smiling",
    "bald": "bald",
    "receeding_hairline": "a receding hairline",
    "long_hair": "long hair",
    "curly_hair": "curly hair",
    "grey_hair": "grey hair",
    "black_hair": "black hair",
    "has_beard": "a beard",
    "patchy_beard": "a patchy beard",
    "has_mustache": "a mustache",
    "well_groomed": "well groomed",
    "has_makeup": "wearing makeup",
    "wearing_glasses": "wearing glasses",
    "wearing_hat": "wearing a hat",
    "clear_skin": "clear skin",
    "dark_circles": "dark circles under the eyes",
    "oily_skin": "oily skin",
    "thick_eyebrow": "thick eyebrows",
    "big_eyes": "big eyes",
    "big_lips": "big lips",
    "sharp_nose": "a sharp nose",
    "mouth_open": "with mouth open",
    "double_chin": "a double chin",
    "veil": "wearing a veil",
    "wrinkle": "wrinkles",
    "chubby": "a chubby face"
}

In [8]:
def get_base_gender(row):
    return "a man" if row["male"] == 1 else "a woman"


def get_age_phrase(row):
    if row["old"] == 1:
        return "elderly"
    elif row["adult"] == 1:
        return "an adult"
    return ""


def get_active_attributes(row, max_attrs=6):
    attrs = []

    for col, desc in ATTRIBUTE_MAP.items():
        if row[col] == 1:
            attrs.append(desc)

    # Limit attributes (important for CLIP)
    if len(attrs) > max_attrs:
        attrs = random.sample(attrs, max_attrs)

    return attrs

In [5]:
def generate_rule_prompt(row):
    base = get_base_gender(row)
    age = get_age_phrase(row)
    attrs = get_active_attributes(row)

    sentence = f"A photo of {base}"

    if age:
        sentence += f", {age}"

    if attrs:
        sentence += " with " + ", ".join(attrs)

    sentence += "."

    return sentence

In [6]:
tqdm.pandas()

df["rule_prompt"] = df.progress_apply(generate_rule_prompt, axis=1)

df[["image_id", "rule_prompt"]].head()

  0%|          | 0/100 [00:00<?, ?it/s]

100%|██████████| 100/100 [00:00<00:00, 19899.91it/s]


,image_id,rule_prompt
0,00001.jpg,"A photo of a man, an adult with wearing a hat,..."
1,00002.jpg,"A photo of a woman, an adult with thick eyebro..."
2,00003.jpg,"A photo of a man, an adult with a mustache, we..."
3,00004.jpg,"A photo of a woman with an attractive face, lo..."
4,00005.jpg,"A photo of a man, an adult with with mouth ope..."


In [ ]:
GROQ_API_KEY = os.getenv("GROQ_API_KEY", "")

In [9]:
def row_to_text(row):
    attrs = get_active_attributes(row, max_attrs=8)
    return ", ".join(attrs)

In [12]:
def build_llm_prompt(row):
    base = get_base_gender(row)
    attr_text = row_to_text(row)

    return f"""
Describe a person's face naturally.

Base: {base}
Attributes: {attr_text}

Rules:
- One sentence only
- Fluent and human-like
- Do not list attributes
"""

In [14]:
def call_llama(prompt, retries=2):
    url = "https://api.groq.com/openai/v1/chat/completions"

    headers = {
        "Authorization": f"Bearer {GROQ_API_KEY}",
        "Content-Type": "application/json"
    }

    data = {
        "model": "llama-3.3-70b-versatile",
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.7
    }

    retryable_status = {429, 500, 502, 503, 504}

    for attempt in range(retries):
        try:
            res = requests.post(url, headers=headers, json=data, timeout=20)
            if res.status_code == 200:
                return res.json()['choices'][0]['message']['content']

            # Do not retry on permanent failures (bad key, forbidden, bad model/payload)
            if res.status_code not in retryable_status:
                return f"ERROR {res.status_code}: {res.text[:160]}"

            if attempt < retries - 1:
                time.sleep(1.5 * (attempt + 1))
        except requests.RequestException as exc:
            if attempt < retries - 1:
                time.sleep(1.5 * (attempt + 1))
            else:
                return f"ERROR REQUEST: {type(exc).__name__}: {str(exc)[:140]}"

    return "ERROR"

In [18]:
llama_prompts = []
df["llama_prompt"] = None

for i, (row_index, row) in enumerate(tqdm(df.iterrows(), total=len(df))):
    prompt = build_llm_prompt(row)
    result = call_llama(prompt)

    llama_prompts.append(result)
    df.at[row_index, "llama_prompt"] = result

    # Save every 500 rows
    if i % 500 == 0:
        df.to_csv("temp_llama.csv", index=False)

df.to_csv("temp_llama.csv", index=False)

  0%|          | 0/100 [00:00<?, ?it/s]

100%|██████████| 100/100 [03:06<00:00,  1.86s/it]


In [ ]:
TOGETHER_API_KEY = os.getenv("TOGETHER_API_KEY", "")

In [20]:
def call_mistral(prompt, retries=2):
    url = "https://api.together.xyz/v1/chat/completions"

    headers = {
        "Authorization": f"Bearer {TOGETHER_API_KEY}",
        "Content-Type": "application/json"
    }

    data = {
        "model": "mistralai/Mistral-7B-Instruct-v0.1",
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.7
    }

    retryable_status = {429, 500, 502, 503, 504}

    for attempt in range(retries):
        try:
            res = requests.post(url, headers=headers, json=data, timeout=20)
            if res.status_code == 200:
                return res.json()['choices'][0]['message']['content']

            # Together billing exhausted: fallback to Groq so generation can continue
            if res.status_code == 402:
                return call_llama(prompt)

            if res.status_code not in retryable_status:
                return f"ERROR {res.status_code}: {res.text[:160]}"

            if attempt < retries - 1:
                time.sleep(1.5 * (attempt + 1))
        except requests.RequestException as exc:
            if attempt < retries - 1:
                time.sleep(1.5 * (attempt + 1))
            else:
                return f"ERROR REQUEST: {type(exc).__name__}: {str(exc)[:140]}"

    return "ERROR"

In [21]:
mistral_prompts = []
df["mistral_prompt"] = None

for i, (row_index, row) in enumerate(tqdm(df.iterrows(), total=len(df))):
    prompt = build_llm_prompt(row)
    result = call_mistral(prompt)

    mistral_prompts.append(result)
    df.at[row_index, "mistral_prompt"] = result

    if i % 500 == 0:
        df.to_csv("temp_mistral.csv", index=False)

df.to_csv("temp_mistral.csv", index=False)

100%|██████████| 100/100 [05:13<00:00,  3.14s/it]


In [22]:
def call_gemma(prompt, retries=2):
    url = "https://api.together.xyz/v1/chat/completions"

    headers = {
        "Authorization": f"Bearer {TOGETHER_API_KEY}",
        "Content-Type": "application/json"
    }

    data = {
        "model": "google/gemma-7b-it",
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.7
    }

    retryable_status = {429, 500, 502, 503, 504}

    for attempt in range(retries):
        try:
            res = requests.post(url, headers=headers, json=data, timeout=20)
            if res.status_code == 200:
                return res.json()['choices'][0]['message']['content']

            # Together billing exhausted: fallback to Groq so generation can continue
            if res.status_code == 402:
                return call_llama(prompt)

            if res.status_code not in retryable_status:
                return f"ERROR {res.status_code}: {res.text[:160]}"

            if attempt < retries - 1:
                time.sleep(1.5 * (attempt + 1))
        except requests.RequestException as exc:
            if attempt < retries - 1:
                time.sleep(1.5 * (attempt + 1))
            else:
                return f"ERROR REQUEST: {type(exc).__name__}: {str(exc)[:140]}"

    return "ERROR"

In [23]:
gemma_prompts = []
df["gemma_prompt"] = None

for i, (row_index, row) in enumerate(tqdm(df.iterrows(), total=len(df))):
    prompt = build_llm_prompt(row)
    result = call_gemma(prompt)

    gemma_prompts.append(result)
    df.at[row_index, "gemma_prompt"] = result

    if i % 500 == 0:
        df.to_csv("temp_gemma.csv", index=False)

df.to_csv("temp_gemma.csv", index=False)

100%|██████████| 100/100 [05:06<00:00,  3.07s/it]


In [24]:
df.to_csv("isgd_prompts.csv", index=False)